# Notebook 2 — Multi-token Word Extraction

Extracts residual stream hidden states at `L_det`, `L_erased`, and `L_mid` for the lowercase, space-prefixed multi-token words used by notebook 1. This is a word-level counterpart to the concept-level extraction notebook.

**Inputs**
- `data/stage1/layer_boundaries.json` → `L_det`, `L_pred`, `L_erased` (written by notebook 1)
- `data/stage1/concept_vocabulary.pkl` — bare deduplicated words from notebook 1b
- `data/stage1/concept_vocabulary_spaced.pkl` — space-prefixed vocabulary from notebook 1b

**Outputs**
- `data/stage1/word_hidden_states_last_token_ldet.npy`    — shape `(n_words, d_model)`
- `data/stage1/word_hidden_states_mean_pool_ldet.npy`
- `data/stage1/word_hidden_states_last_token_lerased.npy`
- `data/stage1/word_hidden_states_mean_pool_lerased.npy`
- `data/stage1/word_hidden_states_last_token_lmid.npy`
- `data/stage1/word_hidden_states_mean_pool_lmid.npy`
- `data/stage1/word_metadata_lower.csv`                   — extraction order and token metadata

**Blocked by:** `1-layer-calibration.ipynb` and `1b-concept-vocabulary.ipynb`


In [ ]:
# parameters
DATA_DIR        = "../../data/stage1"
VOCAB_PKL_BARE  = "../../data/stage1/concept_vocabulary.pkl"
VOCAB_PKL       = "../../data/stage1/concept_vocabulary_spaced.pkl"


In [ ]:
import json
import os
import pickle
import shutil
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
import psutil as _psutil
import torch
from transformer_lens import HookedTransformer

sys.path.insert(0, "../..")
from src.config import MODEL_NAME, TORCH_DTYPE

os.makedirs(DATA_DIR, exist_ok=True)

for _p in (VOCAB_PKL_BARE, VOCAB_PKL):
    if not os.path.exists(_p):
        raise FileNotFoundError(f"{_p} not found. Run 1b-concept-vocabulary.ipynb first.")

_boundaries_path = os.path.join(DATA_DIR, "layer_boundaries.json")
if not os.path.exists(_boundaries_path):
    raise FileNotFoundError(
        f"{_boundaries_path} not found. Run 1-layer-calibration.ipynb first."
    )
with open(_boundaries_path) as _f:
    _boundaries = json.load(_f)
L_DET    = _boundaries["L_det"]
L_PRED   = _boundaries["L_pred"]
L_PRED_PLATEAU = _boundaries.get("L_pred_plateau")
L_PRED_METHOD  = _boundaries.get("L_pred_method", "legacy")
L_ERASED = _boundaries["L_erased"]

print(f"Model:    {MODEL_NAME}")
print(f"L_DET:    {L_DET}")
print(f"L_PRED:   {L_PRED}")
if L_PRED_METHOD == "fallback_last_layer":
    print(f"NOTE: no sustained plateau detected in notebook 1; using last-layer fallback L_PRED={L_PRED}")
elif L_PRED_METHOD != "plateau":
    print(f"NOTE: L_pred_method={L_PRED_METHOD!r}; treating L_PRED={L_PRED} as the effective boundary")
elif L_PRED_PLATEAU is not None:
    print(f"L_PRED_PLATEAU: {L_PRED_PLATEAU}")
print(f"L_ERASED: {L_ERASED}")
print(f"dtype:    {TORCH_DTYPE}")
L_MID = (L_ERASED + L_PRED) // 2
if L_ERASED >= L_PRED:
    raise ValueError(
        f"Degenerate calibration: L_ERASED ({L_ERASED}) >= L_PRED ({L_PRED}). "
        "Re-run notebook 1."
    )
print(f"L_MID:    {L_MID}  (midpoint of [L_ERASED={L_ERASED}, L_PRED={L_PRED}])")

def _ts():
    return datetime.now().strftime("%H:%M:%S")


In [ ]:
import subprocess as _sub
_nb_start = time.time()
try:
    _git_hash = _sub.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=_sub.DEVNULL).decode().strip()
except Exception:
    _git_hash = "unknown"
print(f"[{_ts()}] ═" * 42 + "═")
print(f"[{_ts()}]  Notebook 2 — Multi-token Word Extraction")
print(f"[{_ts()}]  git: {_git_hash}   DATA_DIR: {DATA_DIR}")
print(f"[{_ts()}]  Python {sys.version.split()[0]}  torch {torch.__version__}  CUDA {torch.version.cuda}")
print(f"[{_ts()}]  RAM: {_psutil.virtual_memory().total/1e9:.1f}GB total  {_psutil.virtual_memory().available/1e9:.1f}GB avail")
print(f"[{_ts()}] ═" * 42 + "═")
sys.stdout.flush()


In [ ]:
_cell_start = time.time()
print(f"[{_ts()}] === Load model ==="); sys.stdout.flush()
print(f"[{_ts()}]  Loading {MODEL_NAME} ..."); sys.stdout.flush()
model = HookedTransformer.from_pretrained(MODEL_NAME, dtype=TORCH_DTYPE)
model.eval()
print(f"[{_ts()}]  Model loaded in {time.time()-_cell_start:.0f}s")
print(f"d_model: {model.cfg.d_model}")
_gpu_end = f"  GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB alloc / {torch.cuda.memory_reserved()/1e9:.2f}GB res" if torch.cuda.is_available() else ""
_rss_end = f"  RAM: {_psutil.Process().memory_info().rss/1e9:.2f}GB RSS"
print(f"[{_ts()}]  Cell complete  elapsed {time.time()-_cell_start:.1f}s{_gpu_end}{_rss_end}"); sys.stdout.flush()


In [ ]:
with open(VOCAB_PKL_BARE, "rb") as f:
    vocab_bare = pickle.load(f)
with open(VOCAB_PKL, "rb") as f:
    vocab_spaced = pickle.load(f)

# Mirror notebook 1 exactly: keep only words that are multi-token in both the bare and
# leading-space forms, then lowercase/deduplicate and require the lowercase space-prefixed
# form to remain multi-token.
multi_token_words = [
    w for w in vocab_bare
    if len(model.to_tokens(w)[0].tolist()) >= 3
    and len(model.to_tokens(" " + w)[0].tolist()) >= 3
]
word_names = multi_token_words
word_names_lower = list(dict.fromkeys(w.lower() for w in word_names))
multi_token_words_lower = [
    w for w in word_names_lower
    if len(model.to_tokens(" " + w)[0].tolist()) >= 3
]

word_metadata = pd.DataFrame({
    "word": multi_token_words_lower,
    "spaced_word": [" " + w for w in multi_token_words_lower],
    "n_tokens_spaced": [len(model.to_tokens(" " + w)[0].tolist()) - 1 for w in multi_token_words_lower],
})

word_names_extract = word_metadata["word"].tolist()
n_words = len(word_names_extract)
d_model = model.cfg.d_model

print(f"Vocabulary size:                        {len(vocab_bare)}")
print(f"Multi-token in both forms:              {len(multi_token_words)}")
print(f"Lowercase multi-token (spaced):         {n_words}")
print("Examples (bare):      ", word_names_extract[:3])
print("Examples (spaced):    ", [" " + w for w in word_names_extract[:3]])
print("Token lengths (first):", word_metadata["n_tokens_spaced"].head(5).tolist())


## 2a. Extract hidden states at three calibration landmarks

Three extraction layers span the calibration window established in notebook 1:

| Layer variable | Meaning |
|---|---|
| `L_DET` | Beginning of erasure window — subword tokens still strongly present |
| `L_ERASED` | End of erasure window — subword identity integrated |
| `L_MID` | Midpoint of `[L_ERASED, L_PRED]` — post-erasure concept window |

Both co-primary extraction methods run at each layer:
- **Method A** — last-token hidden state: `h[:, -1, :]`
- **Method B** — mean-pool over all token positions: `h.mean(dim=1)`

Yielding six output arrays. Words are passed as lowercase, space-prefixed strings (e.g. `" carcinoma"`) to match the canonical form used in notebook 1.

Only the three target layers are cached per forward pass (`names_filter`) to keep memory usage low.


In [ ]:
_cell_start = time.time()
print(f"[{_ts()}] === Extract hidden states ({n_words} words) ==="); sys.stdout.flush()
EXTRACTION_LAYERS = {
    "ldet":    L_DET,
    "lerased": L_ERASED,
    "lmid":    L_MID,
}

target_hooks = {
    f"blocks.{L}.hook_resid_post"
    for L in EXTRACTION_LAYERS.values()
}

hidden = {
    (tag, method): np.zeros((n_words, d_model), dtype=np.float32)
    for tag in EXTRACTION_LAYERS
    for method in ("last", "mean")
}

for i, word in enumerate(word_names_extract):
    if i % 200 == 0:
        _el = time.time() - _cell_start
        _rate = f"  ~{_el/i:.3f}s/word" if i > 0 else ""
        print(f"[{_ts()}]  word {i}/{n_words}  elapsed {_el:.0f}s{_rate}"); sys.stdout.flush()

    tokens = model.to_tokens(" " + word)

    try:
        with torch.no_grad():
            _, cache = model.run_with_cache(
                tokens,
                names_filter=lambda name: name in target_hooks
            )
    except Exception as _e:
        print(f"[{_ts()}]    SKIP word {i} {word!r}: {_e}"); sys.stdout.flush()
        continue

    for tag, L in EXTRACTION_LAYERS.items():
        h = cache[f"blocks.{L}.hook_resid_post"][0]
        hidden[(tag, "last")][i] = h[-1].cpu().float().numpy()
        hidden[(tag, "mean")][i] = h.mean(dim=0).cpu().float().numpy()

print(f"Done. Shape per array: ({n_words}, {d_model})")
_gpu_end = f"  GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB alloc / {torch.cuda.memory_reserved()/1e9:.2f}GB res" if torch.cuda.is_available() else ""
_rss_end = f"  RAM: {_psutil.Process().memory_info().rss/1e9:.2f}GB RSS"
print(f"[{_ts()}]  Cell complete  elapsed {time.time()-_cell_start:.1f}s{_gpu_end}{_rss_end}"); sys.stdout.flush()


In [ ]:
_cell_start = time.time()
print(f"[{_ts()}] === Save hidden state arrays ==="); sys.stdout.flush()
_free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
print(f"[{_ts()}]  Disk free: {_free_gb:.1f}GB"); sys.stdout.flush()
for (tag, method), arr in hidden.items():
    method_label = "last_token" if method == "last" else "mean_pool"
    fname = f"word_hidden_states_{method_label}_{tag}.npy"
    path  = os.path.join(DATA_DIR, fname)
    np.save(path, arr)
    _sz_mb = os.path.getsize(path) / 1e6
    print(f"Saved: {path}  ({_sz_mb:.1f} MB)")

_meta_path = os.path.join(DATA_DIR, "word_metadata_lower.csv")
word_metadata.to_csv(_meta_path, index=False)
print(f"Saved: {_meta_path}")
_rss_end = f"  RAM: {_psutil.Process().memory_info().rss/1e9:.2f}GB RSS"
print(f"[{_ts()}]  Cell complete  elapsed {time.time()-_cell_start:.1f}s{_rss_end}"); sys.stdout.flush()


## Verification

Checks that all six hidden state files were written with the correct shape and are non-zero, and that `word_metadata_lower.csv` was written. A `FAIL` on any row means the extraction loop did not complete successfully for that array.


In [ ]:
# Verify all six hidden state files
_expected = (n_words, d_model)
_all_ok = True

print(f"Expected shape: {_expected}")
print(f"Extraction layers: { {k: v for k, v in EXTRACTION_LAYERS.items()} }")
print()

for _tag in EXTRACTION_LAYERS:
    for _ml in ("last_token", "mean_pool"):
        _fname = f"word_hidden_states_{_ml}_{_tag}.npy"
        _path  = os.path.join(DATA_DIR, _fname)
        if not os.path.exists(_path):
            print(f"  MISSING  {_fname}")
            _all_ok = False
            continue
        _arr = np.load(_path)
        _shape_ok   = _arr.shape == _expected
        _nonzero_ok = bool(_arr.any())
        _status = "OK  " if (_shape_ok and _nonzero_ok) else "FAIL"
        print(f"  {_status}  {_fname}  shape={_arr.shape}  non-zero={_nonzero_ok}")
        if not (_shape_ok and _nonzero_ok):
            _all_ok = False

_meta_path = os.path.join(DATA_DIR, "word_metadata_lower.csv")
if os.path.exists(_meta_path):
    _meta = pd.read_csv(_meta_path)
    _meta_ok = len(_meta) == n_words
    _status = "OK  " if _meta_ok else "FAIL"
    print(f"  {_status}  word_metadata_lower.csv  rows={len(_meta)}")
    if not _meta_ok:
        _all_ok = False
else:
    print("  MISSING  word_metadata_lower.csv")
    _all_ok = False

print()
if _all_ok:
    print("All word-level files verified.")
else:
    raise AssertionError("Verification failed — check output above and re-run extraction cells.")

_rss_end = f"  RAM: {_psutil.Process().memory_info().rss/1e9:.2f}GB RSS"
print(f"[{_ts()}]  Notebook complete — total {time.time()-_nb_start:.0f}s elapsed{_rss_end}"); sys.stdout.flush()
